# Season EPA and Breakdown EPAs

This notebook loads the local Statbotics team-year cache, filters it to the season year you enter, optionally merges local match counts from `frc_matches_<year>.csv`, and writes a team-level CSV.

It does not call the Statbotics API at runtime.

In [1]:
import ast
import re
from pathlib import Path

import pandas as pd

CACHE_PATH = Path("statbotics_team_years_all.csv")


def resolve_year_and_match_csv():
    raw = input("Enter season year or frc_matches_<year>.csv path [latest available]: ").strip()
    if not raw:
        if not CACHE_PATH.exists():
            raise FileNotFoundError("Missing statbotics_team_years_all.csv in the current directory.")
        cache_years = pd.read_csv(CACHE_PATH, usecols=["year"])
        year = int(cache_years["year"].max())
        match_csv = Path(f"frc_matches_{year}.csv")
        return year, match_csv if match_csv.exists() else None

    if raw.isdigit():
        year = int(raw)
        match_csv = Path(f"frc_matches_{year}.csv")
        return year, match_csv if match_csv.exists() else None

    match_csv = Path(raw)
    if not match_csv.exists():
        raise FileNotFoundError(f"Could not find match CSV: {match_csv}")

    year_match = re.search(r"(\d{4})", match_csv.stem)
    if year_match:
        year = int(year_match.group(1))
    else:
        probe = pd.read_csv(match_csv, nrows=5, low_memory=False)
        year = int(pd.to_numeric(probe.get("year"), errors="coerce").dropna().iloc[0])
    return year, match_csv


def parse_team_keys(value):
    if isinstance(value, list):
        return [str(item) for item in value]
    if value is None or pd.isna(value):
        return []
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        try:
            parsed = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return [value]
        if isinstance(parsed, list):
            return [str(item) for item in parsed]
    return []


def team_number_from_key(team_key):
    if isinstance(team_key, str):
        match = re.search(r"(\d+)", team_key)
        if match:
            return int(match.group(1))
    return int(team_key)


def load_match_counts(match_csv):
    if match_csv is None:
        return pd.Series(dtype="int64")

    df = pd.read_csv(match_csv, low_memory=False)
    if "alliances.red.score" not in df.columns or "alliances.blue.score" not in df.columns:
        raise ValueError(f"{match_csv} does not look like a season match CSV.")

    counts = {}
    for _, row in df.iterrows():
        red_score = pd.to_numeric(pd.Series([row.get("alliances.red.score")]), errors="coerce").iloc[0]
        blue_score = pd.to_numeric(pd.Series([row.get("alliances.blue.score")]), errors="coerce").iloc[0]
        if pd.isna(red_score) or pd.isna(blue_score):
            continue

        for color in ("red", "blue"):
            for team_key in parse_team_keys(row.get(f"alliances.{color}.team_keys")):
                team_number = team_number_from_key(team_key)
                counts[team_number] = counts.get(team_number, 0) + 1

    return pd.Series(counts, dtype="int64")


In [2]:
if not CACHE_PATH.exists():
    raise FileNotFoundError("Missing statbotics_team_years_all.csv in the current directory.")

year, match_csv = resolve_year_and_match_csv()
cache = pd.read_csv(CACHE_PATH, low_memory=False)
year_df = cache.loc[cache["year"] == year].copy()
if year_df.empty:
    raise ValueError(f"No Statbotics rows were found for year {year}.")

match_counts = load_match_counts(match_csv)
year_df["team_number"] = year_df["team"].astype(int)
year_df["team_key"] = year_df["team_number"].map(lambda team: f"frc{team}")

if not match_counts.empty:
    year_df["matches_played_local"] = year_df["team_number"].map(match_counts).fillna(0)
else:
    year_df["matches_played_local"] = 0

year_df["matches_played_local"] = pd.to_numeric(year_df["matches_played_local"], errors="coerce").fillna(0).astype(int)
year_df["record_count"] = pd.to_numeric(year_df["record_count"], errors="coerce").fillna(0).astype(int)
year_df["matches_played"] = year_df["record_count"]
year_df["matches_played_delta"] = year_df["matches_played_local"] - year_df["record_count"]

year_df["epa_end"] = year_df["epa_total_points_mean"]
year_df["unitless_epa_end"] = year_df["epa_unitless"]
year_df["norm_epa_end"] = year_df["epa_norm"]
year_df["epa_start"] = year_df["epa_stats_start"]
year_df["epa_pre_champs"] = year_df["epa_stats_pre_champs"]
year_df["epa_max"] = year_df["epa_stats_max"]
year_df["total_epa_rank"] = year_df["epa_rank_total_rank"]
year_df["country_epa_rank"] = year_df["epa_rank_country_rank"]
year_df["state_epa_rank"] = year_df["epa_rank_state_rank"]
year_df["district_epa_rank"] = year_df["epa_rank_district_rank"]

preferred = [
    "team_number",
    "team_key",
    "year",
    "name",
    "country",
    "state",
    "district",
    "rookie_year",
    "competing",
    "district_points",
    "district_rank",
    "record_wins",
    "record_losses",
    "record_ties",
    "record_count",
    "record_winrate",
    "matches_played",
    "matches_played_local",
    "matches_played_delta",
    "epa_end",
    "epa_total_points_mean",
    "epa_total_points_sd",
    "unitless_epa_end",
    "epa_unitless",
    "norm_epa_end",
    "epa_norm",
    "epa_conf_low",
    "epa_conf_high",
    "epa_start",
    "epa_pre_champs",
    "epa_max",
    "total_epa_rank",
    "epa_rank_total_percentile",
    "epa_rank_total_team_count",
    "country_epa_rank",
    "epa_rank_country_percentile",
    "epa_rank_country_team_count",
    "state_epa_rank",
    "epa_rank_state_percentile",
    "epa_rank_state_team_count",
    "district_epa_rank",
    "epa_rank_district_percentile",
    "epa_rank_district_team_count",
]

other_cols = [column for column in year_df.columns if column not in preferred]
year_df = year_df[preferred + other_cols].sort_values(["epa_end", "team_number"], ascending=[False, True]).reset_index(drop=True)

output_path = Path(f"statbotics_team_year_{year}.csv")
year_df.to_csv(output_path, index=False)

mismatch_count = int((year_df["matches_played_delta"] != 0).sum())
print(f"Loaded {len(year_df):,} Statbotics team-year rows for {year}")
if match_csv is not None:
    print(f"Matched local season CSV: {match_csv.name}")
print(f"Wrote {output_path}")
if mismatch_count:
    print(f"Warning: {mismatch_count} teams had a local match count different from Statbotics record_count")
year_df.head(20)


Loaded 3,702 Statbotics team-year rows for 2025
Matched local season CSV: frc_matches_2025.csv
Wrote statbotics_team_year_2025.csv


,team_number,team_key,year,name,country,state,district,rookie_year,competing,district_points,...,breakdown_auto_fuel,breakdown_auto_tower,breakdown_transition_fuel,breakdown_first_shift_fuel,breakdown_second_shift_fuel,breakdown_teleop_fuel,breakdown_endgame_fuel,breakdown_endgame_tower,breakdown_total_fuel,breakdown_total_tower
0,2056,frc2056,2025,OP Robotics,Canada,ON,ont,2007.0,NaN,392.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2910,frc2910,2025,Jack in the Bot,USA,WA,pnw,2009.0,NaN,356.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1323,frc1323,2025,MadTown Robotics,USA,CA,NaN,2004.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1690,frc1690,2025,Orbit,Israel,NaN,isr,2005.0,NaN,364.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1678,frc1678,2025,Citrus Circuits,USA,CA,NaN,2005.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,118,frc118,2025,Robonauts,USA,TX,fit,1997.0,NaN,377.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2481,frc2481,2025,Roboteers,USA,IL,NaN,2008.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,5940,frc5940,2025,BREAD,USA,CA,NaN,2016.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1796,frc1796,2025,RoboTigers,USA,NY,NaN,2006.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,4678,frc4678,2025,CyberCavs,Canada,ON,ont,2013.0,NaN,380.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
